# Módulo 02 — Exploração, Filtros e Agrupamento de Tabelas com `Pandas`

## 1. O que é o `Pandas`?

Enquanto o `NumPy` trabalha com matrizes puramente numéricas, na vida real os dados experimentais vêm em **planilhas** (arquivos `.csv` ou Excel) que misturam códigos de lote, nomes, categorias e grandezas físicas.

O **`Pandas`** é a biblioteca padrão da indústria para carregar, inspecionar, filtrar e resumir tabelas de dados em Python.
- **`DataFrame` (Tabela):** Estrutura bidimensional onde cada **linha** é uma amostra/observação (ex.: um passageiro ou um ensaio de laboratório) e cada **coluna** é uma **feature** (*atributo/variável*, como idade, pressão ou classe).
- **`Series` (Coluna):** Uma única coluna extraída de um `DataFrame`.

Vamos carregar o clássico conjunto de dados do naufrágio do **Titanic (1912)** armazenado em `arquivos-csv/titanic.csv`:

In [ ]:
# Importa o Pandas com o apelido universal 'pd'
import pandas as pd

# Lê o arquivo CSV usando o caminho relativo correto a partir da pasta 'documentacao-didatica/'
df = pd.read_csv("arquivos-csv/titanic.csv")

# Verifica quantas linhas (passageiros) e colunas (variáveis) foram carregadas
print(f"Dimensões da tabela: {df.shape[0]} linhas x {df.shape[1]} colunas")

**Análise:** Carregamos `1309` registros com `15` colunas em uma fração de segundo!

## 2. Inspeção Rápida: `head()`, `tail()`, `info()` e `describe()`

Antes de fazer qualquer conta em engenharia, inspecionamos a cara dos dados:
- `df.head(n)`: Exibe as **`n` primeiras linhas** da tabela (padrão `5`).
- `df.tail(n)`: Exibe as **`n` últimas linhas** da tabela.

In [ ]:
# Visualizando as 5 primeiras linhas do DataFrame
print("--- Primeiras 5 linhas ---")
print(df.head(5))

# Visualizando as 3 últimas linhas do DataFrame
print("\n--- Últimas 3 linhas ---")
print(df.tail(3))

Agora vamos pedir o "raio-X estrutural" (`df.info()`) e o "resumo estatístico" (`df.describe()`):
- `df.info()`: Mostra o nome de cada coluna, quantos valores preenchidos existem e o **tipo de dado (`dtype`)**: `int64` (inteiro), `float64` (decimal) ou `object`/`str` (texto).
- `df.describe()`: Calcula automaticamente contagem (`count`), média (`mean`), desvio padrão (`std`), mínimo (`min`), quartis (`25%`, `50%` = mediana, `75%`) e máximo (`max`) das colunas numéricas.

In [ ]:
# Exibe tipos de cada coluna e uso de memória
print("--- Estrutura da Tabela (info) ---")
df.info()

# Exibe estatísticas descritivas das colunas reconhecidas como numéricas
print("\n--- Resumo Estatístico (describe) ---")
print(df.describe())

**Olhar Crítico de Engenheiro(a):** Repare em um detalhe curioso na saída acima! Por que `age` (idade) e `fare` (tarifa paga) **não apareceram** no `df.describe()` e estão listadas como texto (`object`) no `df.info()`?
Porque no arquivo bruto alguém registrou dados desconhecidos usando o caractere `'?'`! No **Módulo 03**, aprenderemos a limpar essa sujeira. Por enquanto, vamos aprender a selecionar, filtrar e agrupar dados!

---

## 3. Selecionando Colunas e Filtrando Linhas

- Selecionar **uma coluna** (`Series`): `df['pclass']`
- Selecionar **várias colunas** (`DataFrame` menor): `df[['name', 'sex', 'pclass', 'survived']]`
- **Filtrar linhas por condição:** `df[df['pclass'] == 1]` (retorna apenas passageiros da 1ª classe). Para combinar condições, usamos `&` (*E lógico*) e `|` (*OU lógico*) com cada condição entre parênteses.

In [ ]:
# Filtrando apenas mulheres ('female') que viajaram na 1ª classe (pclass == 1)
mulheres_1a_classe = df[(df["sex"] == "female") & (df["pclass"] == 1)]

print(f"Total de mulheres na 1ª classe: {len(mulheres_1a_classe)}")
print("Taxa de sobrevivência desse grupo: {:.2f}%".format(
    mulheres_1a_classe["survived"].mean() * 100.0
))

**Análise:** Como a coluna `survived` vale `1` (sobreviveu) ou `0` (não sobreviveu), calcular a **média** (`.mean()`) de zeros e uns fornece matematicamente a **fração/proporção de sobreviventes** (`96,53%` das mulheres da 1ª classe sobreviveram!).

---

## 4. Agrupamento Inteligente (`groupby`) e Exportação (`to_csv`)

E se quisermos calcular essa estatística para **todas as classes ou categorias de uma só vez** (como numa Tabela Dinâmica do Excel)? Usamos a tríade:
1. `df.groupby('coluna_grupo')['coluna_alvo'].mean()`: Divide a tabela por grupo e calcula a média de cada grupo.
2. `.reset_index()`: Transforma o resultado agrupado de volta em um `DataFrame` plano com colunas normais.
3. `.rename(columns={'antigo': 'novo'})`: Dá um nome claro à coluna calculada.
4. `.to_csv('arquivo.csv', index=False)`: Salva um novo CSV sem gravar a coluna numérica de índice lateral.

In [ ]:
# Agrupando passageiros pela classe do bilhete (1ª, 2ª e 3ª classe)
resumo_classe = (
    df.groupby("pclass")["survived"]  # Agrupa por 'pclass' e olha para 'survived'
    .mean()                           # Calcula a fração média de sobreviventes em cada classe
    .reset_index()                    # Converte o índice agrupado em coluna normal
    .rename(columns={"survived": "taxa_sobrevivencia_pct"})  # Renomeia a coluna de saída
)

# Multiplica por 100 para exibir em porcentagem (%)
resumo_classe["taxa_sobrevivencia_pct"] = resumo_classe["taxa_sobrevivencia_pct"] * 100.0

print("Taxa de sobrevivência por classe socioeconômica:")
print(resumo_classe.round(2))

**Interpretação dos Dados:** O resultado revela o forte contraste operacional do naufrágio: cerca de **61,9%** dos passageiros da 1ª classe sobreviveram, contra **43,0%** na 2ª classe e apenas **25,5%** na 3ª classe. Essa diferença explica por que `pclass` será uma *feature* decisiva quando treinarmos nossa Rede Neural mais adiante!

---

## 5. Exercícios Práticos de Fixação

### Exercício 1: Contagem por Porto de Embarque (`value_counts`)
A coluna `embarked` indica o porto onde cada passageiro subiu no navio (`S` = Southampton, `C` = Cherbourg, `Q` = Queenstown). Use `df['embarked'].value_counts()` para descobrir quantos passageiros embarcaram em cada porto.

In [ ]:
# Escreva seu código aqui

<details>
<summary><b>Clique aqui para ver o gabarito comentado</b></summary>

```python
import pandas as pd

df = pd.read_csv("arquivos-csv/titanic.csv")

# value_counts() conta automaticamente a frequência de cada valor único na coluna
contagem_portos = df["embarked"].value_counts()
print("Passageiros por porto de embarque:")
print(contagem_portos)
```

</details>

### Exercício 2: Comparação de Sobrevivência por Gênero (`groupby`)
Agrupe o `DataFrame` `df` pela coluna `sex` e calcule a porcentagem média de sobrevivência (`survived * 100`) para homens (`male`) e mulheres (`female`).

In [ ]:
# Escreva seu código aqui

<details>
<summary><b>Clique aqui para ver o gabarito comentado</b></summary>

```python
import pandas as pd

df = pd.read_csv("arquivos-csv/titanic.csv")

# Agrupa por gênero, tira a média de 'survived' e converte para porcentagem
sobrevivencia_genero = df.groupby("sex")["survived"].mean().reset_index()
sobrevivencia_genero["survived"] = sobrevivencia_genero["survived"] * 100.0
sobrevivencia_genero = sobrevivencia_genero.rename(columns={"survived": "sobrevivencia_pct"})

print(sobrevivencia_genero.round(2))
```

</details>

### Exercício 3: Criando uma Nova Feature (Tamanho da Família)
Em Aprendizado de Máquina, criar colunas novas combinando dados brutos chama-se **Engenharia de Atributos (*Feature Engineering*)**. Some `sibsp` (irmãos/cônjuges a bordo) + `parch` (pais/filhos a bordo) + `1` (o próprio passageiro) em uma nova coluna `tamanho_familia` e imprima quantos passageiros viajavam completamente sozinhos (`tamanho_familia == 1`).

In [ ]:
# Escreva seu código aqui

<details>
<summary><b>Clique aqui para ver o gabarito comentado</b></summary>

```python
import pandas as pd

df = pd.read_csv("arquivos-csv/titanic.csv")

# Soma coluna a coluna (vetorizada, igual ao NumPy!) para criar a nova feature
df["tamanho_familia"] = df["sibsp"] + df["parch"] + 1

# Filtra quem tinha tamanho_familia igual a 1
viajantes_solo = df[df["tamanho_familia"] == 1]
print(f"Passageiros viajando sozinhos: {len(viajantes_solo)} de {len(df)}")
```

</details>

---

## Próximo Passo

Já sabemos explorar, filtrar e agrupar tabelas limpas. Mas como lidar com os caracteres `'?'`, textos infiltrados em colunas numéricas e sensores falhos (`NaN`) que travam qualquer Rede Neural? Vamos resolver isso no **[Módulo 03 — Pandas: Limpeza e Tratamento de Dados Reais](03_pandas_limpeza_dados_reais.ipynb)**!